# Week 1 exercise — Education research assistant (OpenRouter)

This notebook builds a small **tutor-style** chatbot that uses **tool calling**: the model picks **Wikipedia**, **NewsAPI** (optional), or **DuckDuckGo instant answer** depending on the question.

**LLM provider:** [OpenRouter](https://openrouter.ai/) using the **`openai.OpenAI` client** with `base_url="https://openrouter.ai/api/v1"` (OpenAI-compatible API).



In [1]:
from __future__ import annotations

import json
import os
from typing import Any

import httpx
import wikipedia

from openai import OpenAI
import gradio as gr
from decouple import config


OPENROUTER_MODEL = "openai/gpt-4o-mini"
OPEN_ROUTER_API_KEY = config("OPEN_ROUTER_API_KEY")
NEWS_API_KEY = config("NEWS_API_KEY")


client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPEN_ROUTER_API_KEY,)


print("Model:", OPENROUTER_MODEL)

In [6]:
# --- Tool implementations (called by the LLM via function calling) ---

wikipedia.set_lang("en")


def wikipedia_summary(topic: str, sentences: int = 4) -> dict[str, Any]:
    """Encyclopedic overview; use for stable facts, history, definitions of notable topics."""
    topic = (topic or "").strip()
    if not topic:
        return {"error": "empty topic"}
    try:
        page = wikipedia.page(topic, auto_suggest=True)
        summary = wikipedia.summary(topic, sentences=sentences, auto_suggest=True)
        return {
            "title": page.title,
            "url": page.url,
            "summary": summary,
        }
    except wikipedia.DisambiguationError as e:
        opts = e.options[:8]
        return {
            "error": "disambiguation",
            "message": str(e),
            "options": opts,
        }
    except Exception as e:
        return {"error": type(e).__name__, "message": str(e)}


def news_search(query: str, page_size: int = 5) -> dict[str, Any]:
    """Recent news articles; use when the user asks about current events, 'today', 'this week', breaking news."""
    api_key = NEWS_API_KEY
    if not api_key:
        return {
            "error": "NEWS_API_KEY not set",
            "hint": "Add NEWS_API_KEY from newsapi.org to .env to enable this tool.",
        }
    query = (query or "").strip()
    if not query:
        return {"error": "empty query"}
    page_size = max(1, min(page_size, 10))
    url = "https://newsapi.org/v2/everything"
    params = {
        "q": query,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": page_size,
        "apiKey": api_key,
    }
    try:
        r = httpx.get(url, params=params, timeout=30.0)
        data = r.json()
        if data.get("status") != "ok":
            return {"error": data.get("message", "newsapi error"), "raw": data}
        articles = []
        for a in data.get("articles", []):
            articles.append(
                {
                    "title": a.get("title"),
                    "url": a.get("url"),
                    "source": (a.get("source") or {}).get("name"),
                    "publishedAt": a.get("publishedAt"),
                    "description": a.get("description"),
                }
            )
        return {"query": query, "articles": articles}
    except Exception as e:
        return {"error": type(e).__name__, "message": str(e)}


def duckduckgo_instant_answer(query: str) -> dict[str, Any]:
    """This is the tool calling function. Quick instant-answer box (abstract + URL); use for short facts or when Wikipedia is unclear. No API key."""
    query = (query or "").strip()
    if not query:
        return {"error": "empty query"}
    try:
        r = httpx.get(
            "https://api.duckduckgo.com/",
            params={"q": query, "format": "json", "no_html": 1},
            timeout=20.0,
        )
        d = r.json()
        out = {
            "query": query,
            "abstract": d.get("Abstract") or "",
            "abstract_url": d.get("AbstractURL") or "",
            "heading": d.get("Heading") or "",
        }
        topics = d.get("RelatedTopics") or []
        snippets = []
        for t in topics[:5]:
            if isinstance(t, dict) and "Text" in t:
                snippets.append({"text": t.get("Text"), "url": t.get("FirstURL")})
        out["related"] = snippets
        if not out["abstract"] and not snippets:
            out["note"] = "No instant answer; try wikipedia_summary with a clearer topic."
        return out
    except Exception as e:
        return {"error": type(e).__name__, "message": str(e)}

In [3]:

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "wikipedia_summary",
            "description": (
                "Use for encyclopedic background: definitions, history, science concepts, "
                "biographies of well-known topics. Prefer this when the question is not time-sensitive."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "Topic or article title to look up on Wikipedia.",
                    },
                    "sentences": {
                        "type": "integer",
                        "description": "Number of sentences in the summary (default 4, max ~8).",
                    },
                },
                "required": ["topic"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "news_search",
            "description": (
                "Use for recent news: 'latest', 'today', 'this week', breaking events, "
                "or when freshness matters. Requires NEWS_API_KEY in the environment."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "News search query."},
                    "page_size": {
                        "type": "integer",
                        "description": "Number of articles to return (1–10).",
                    },
                },
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "duckduckgo_instant_answer",
            "description": (
                "Use for quick factual blurbs or when Wikipedia might miss or disambiguate badly. "
                "No API key; results can be sparse."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query."},
                },
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
]

TOOL_REGISTRY = {
    "wikipedia_summary": wikipedia_summary,
    "news_search": news_search,
    "duckduckgo_instant_answer": duckduckgo_instant_answer,
}


def handle_tool_calls(tool_calls) -> list[dict]:
    results = []
    for tc in tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments or "{}")
        print(f"Tool: {name}({args})", flush=True)
        fn = TOOL_REGISTRY.get(name)
        payload = fn(**args) if fn else {"error": f"unknown tool {name}"}
        results.append(
            {
                "role": "tool",
                "content": json.dumps(payload),
                "tool_call_id": tc.id,
            }
        )
    return results


SYSTEM_PROMPT = """You are a helpful educational assistant for learners.

- Choose tools wisely: use news_search for timely/current events; use wikipedia_summary for stable concepts;
  use duckduckgo_instant_answer for quick facts when appropriate.
- Ground your answers in tool results. Cite titles and URLs when tools return them.
- If a tool returns an error or empty data, say so honestly—do not invent sources.
- Keep explanations clear and appropriate for a student audience.
"""

In [4]:
def chat(message: str, history: list) -> str:
    """Gradio ChatInterface (type='messages'): history is list of {role, content} dicts."""
    messages: list[dict] = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(history)
    messages.append({"role": "user", "content": message})

    done = False
    response = None
    while not done:
        response = client.chat.completions.create(
            model=OPENROUTER_MODEL,
            messages=messages,
            tools=TOOLS,
        )
        choice = response.choices[0]
        if choice.finish_reason == "tool_calls" and choice.message.tool_calls:
            msg = choice.message
            tool_results = handle_tool_calls(msg.tool_calls)
            messages.append(msg)
            messages.extend(tool_results)
        else:
            done = True

    return response.choices[0].message.content or ""

In [5]:
# Launch UI (blocks until you stop the kernel)
gr.ChatInterface(chat, type="messages", title="Education research (OpenRouter)").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
